# Configure Lakehouse Table Properties

Loops through all Delta tables in a lakehouse and applies table-level properties:
- `delta.autoOptimize.autoCompact` — triggers compaction after each write when fragmentation is detected
- `delta.autoOptimize.optimizeWrite` — coalesces small files before writing (enable for append/streaming, disable for overwrite)

Set as table properties (not session configs) so they persist across all sessions and jobs.

Docs:
- https://learn.microsoft.com/en-us/fabric/data-engineering/delta-optimization-and-v-order
- https://learn.microsoft.com/en-us/fabric/fundamentals/table-maintenance-optimization

## Parameters

Set the target workspace and lakehouse by name. Leave either blank to default to the workspace
and lakehouse attached to this notebook.

`apply_optimize_write` should be `True` for append/streaming tables that produce many small files,
and `False` for tables written by CopyJob full overwrites — overwrite mode already produces
well-sized files and the extra overhead isn't worth it.

Set `dry_run = True` to preview what would change without modifying any tables.

In [ ]:
# Pass workspace and lakehouse by name — no need to look up GUIDs manually.
# Leave either blank ("") to default to the workspace/lakehouse attached to this notebook.
workspace_name       = ""    # e.g. "My Fabric Workspace"
lakehouse_name       = ""    # e.g. "gold_lakehouse"
apply_optimize_write = True  # Set False for overwrite-only tables (e.g. CopyJob full overwrite)
dry_run              = False # Set True to preview changes without applying them

## Setup

Resolves the workspace and lakehouse names to their GUIDs using `sempy.fabric`.
This avoids having to manually look up IDs in the Fabric portal URL.

If `workspace_name` or `lakehouse_name` are left blank, the notebook defaults to the
workspace and lakehouse it is attached to.

In [ ]:
import sempy.fabric as fabric
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

# Resolve workspace — defaults to the attached notebook workspace if blank
if workspace_name:
    workspace_id = fabric.resolve_workspace_id(workspace_name)
else:
    workspace_id   = fabric.get_workspace_id()
    workspace_name = fabric.resolve_workspace_name(workspace_id)

# Resolve lakehouse ID by name within the workspace
if lakehouse_name:
    items_df = fabric.list_items(workspace=workspace_id, item_type="Lakehouse")
    matched  = items_df[items_df["Display Name"] == lakehouse_name]
    if matched.empty:
        raise ValueError(f"Lakehouse '{lakehouse_name}' not found in workspace '{workspace_name}'")
    lakehouse_id = matched["Id"].iloc[0]
else:
    lakehouse_id   = fabric.get_lakehouse_id()
    lakehouse_name = fabric.resolve_item_name(lakehouse_id, workspace=workspace_id)

base_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

print(f"Workspace      : {workspace_name} ({workspace_id})")
print(f"Lakehouse      : {lakehouse_name} ({lakehouse_id})")
print(f"Base path      : {base_path}")
print(f"Optimize write : {apply_optimize_write}")
print(f"Dry run        : {dry_run}")
print(f"Run started    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Discover Tables

Lists all folders under the lakehouse `/Tables` directory using `mssparkutils.fs.ls`.
Each folder represents a Delta table.

`mssparkutils.fs.ls` is preferred over querying the SQL catalog because it works
reliably regardless of how the table was originally created (Copy Activity, notebook,
Dataflow Gen2, etc.). Internal Delta folders prefixed with `_` or `.` are excluded.

In [ ]:
from notebookutils import mssparkutils

# List all items in the Tables directory
table_items = mssparkutils.fs.ls(base_path)

table_paths = [
    item for item in table_items
    if item.isDir
    and not item.name.startswith("_")
    and not item.name.startswith(".")
]

print(f"Found {len(table_paths)} tables:")
for t in table_paths:
    print(f"  - {t.name}")

## Apply Table Properties

Loops through each discovered table and applies the target `TBLPROPERTIES` via `ALTER TABLE`.

Setting properties at the **table level** (rather than via `spark.conf.set`) ensures they
persist across all sessions and jobs that write to the table — session configs are lost
when the Spark session ends.

The loop:
1. Verifies the folder is a valid Delta table before proceeding
2. Reads the existing `TBLPROPERTIES` to capture the before state
3. Only applies changes where the current value differs from the target — avoids unnecessary writes
4. Catches errors per-table so a single failure does not stop the run

`dry_run = True` previews what would change without modifying any tables.

In [ ]:
results = []

for item in table_paths:
    table_name = item.name
    table_path = f"{base_path}/{table_name}"
    result = {
        "table":                 table_name,
        "status":                None,
        "auto_compact_before":   None,
        "optimize_write_before": None,
        "changes_applied":       [],
        "error":                 None
    }

    try:
        if not DeltaTable.isDeltaTable(spark, table_path):
            result["status"] = "SKIPPED"
            result["error"]  = "Not a Delta table"
            results.append(result)
            print(f"[SKIPPED] {table_name} — not a Delta table")
            continue

        # Read current properties
        props_df = spark.sql(f"SHOW TBLPROPERTIES delta.`{table_path}`")
        props    = {row["key"]: row["value"] for row in props_df.collect()}

        current_auto_compact   = props.get("delta.autoOptimize.autoCompact",   "not set")
        current_optimize_write = props.get("delta.autoOptimize.optimizeWrite",  "not set")

        result["auto_compact_before"]   = current_auto_compact
        result["optimize_write_before"] = current_optimize_write

        # Build target properties
        # autoCompact is always applied. optimizeWrite is conditional on the parameter.
        properties_to_set = {"delta.autoOptimize.autoCompact": "true"}
        if apply_optimize_write:
            properties_to_set["delta.autoOptimize.optimizeWrite"] = "true"

        # Only change what actually needs changing
        changes_needed = {
            k: v for k, v in properties_to_set.items()
            if props.get(k) != v
        }

        if not changes_needed:
            result["status"] = "NO CHANGE"
            print(f"[NO CHANGE] {table_name} — properties already set correctly")

        elif dry_run:
            result["status"]          = "DRY RUN"
            result["changes_applied"] = list(changes_needed.keys())
            print(f"[DRY RUN] {table_name} — would set: {changes_needed}")

        else:
            props_sql = ", ".join([f"'{k}' = '{v}'" for k, v in changes_needed.items()])
            alter_sql = f"ALTER TABLE delta.`{table_path}` SET TBLPROPERTIES ({props_sql})"
            spark.sql(alter_sql)

            result["status"]          = "UPDATED"
            result["changes_applied"] = list(changes_needed.keys())
            print(f"[UPDATED] {table_name} — set: {changes_needed}")

    except Exception as e:
        result["status"] = "ERROR"
        result["error"]  = str(e)
        print(f"[ERROR] {table_name} — {e}")

    results.append(result)

## Summary

Displays a final run summary showing each table, its status, the property values
before the run, and which changes were applied. Use this to confirm all tables were
updated and to investigate any errors.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

summary_schema = StructType([
    StructField("table",                 StringType(), True),
    StructField("status",                StringType(), True),
    StructField("auto_compact_before",   StringType(), True),
    StructField("optimize_write_before", StringType(), True),
    StructField("changes_applied",       StringType(), True),
    StructField("error",                 StringType(), True),
])

summary_rows = [
    (
        r["table"],
        r["status"],
        r["auto_compact_before"],
        r["optimize_write_before"],
        ", ".join(r["changes_applied"]) if r["changes_applied"] else "",
        r["error"] or ""
    )
    for r in results
]

summary_df = spark.createDataFrame(summary_rows, schema=summary_schema)

print(f"\n{'='*60}")
print(f"RUN COMPLETE — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Workspace : {workspace_name}")
print(f"Lakehouse : {lakehouse_name}")
print(f"{'='*60}")
print(f"  Total tables : {len(results)}")
print(f"  Updated      : {sum(1 for r in results if r['status'] == 'UPDATED')}")
print(f"  No change    : {sum(1 for r in results if r['status'] == 'NO CHANGE')}")
print(f"  Dry run      : {sum(1 for r in results if r['status'] == 'DRY RUN')}")
print(f"  Skipped      : {sum(1 for r in results if r['status'] == 'SKIPPED')}")
print(f"  Errors       : {sum(1 for r in results if r['status'] == 'ERROR')}")

display(summary_df)